# Usage Instructions

## SLS

If on SystemLink server, leave as-is and deploy as an analysis routine set to run once daily. (The actual wall time isn't important.) To retrieve usage data, run the accomonying "get-sls-usage-data-file" notebook and download the resulting CSV for later analysis.

## SLE

If on SystemLink Enterprise, ensure the `WORKSPACE_TO_USE` constant in the `EnterpriseUsageTracking` class is set to the id of the desired workspace for storage of the usage data dataframe. Leaving as `None` will use the Default workspace. After that deploy as a Scheduled daily routine. Again, wall time isn't important, so long as it is consistent. To retrieve usage data, go to the "Dataframes" tab in SLE and export the "UsageCalculationData" to CSV for later analysis.

In [ ]:
import csv
import json
import os
import re

from datetime import datetime
from typing import Any, Dict, List, Optional, Set

import pandas as pd

try:  # assume we're in SLE or SLS with the clients installed
    from nisystemlink.clients.dataframe import DataFrameClient
    from nisystemlink.clients.dataframe.models import (
        QueryTablesRequest,
        CreateTableRequest,
        Column,
        DataType,
        ColumnType,
        DataFrame,
        AppendTableDataRequest,
        ModifyTableRequest,
        ColumnMetadataPatch,
    )
except ImportError:  # we're in SLS
    pass


class UsageTracking:
    """Base class for storing and interacting with daily user tracking data."""

    NUM_TRACKED_USERS = 1000
    TIME_FORMAT = "%Y-%m-%dT%H:%M:%S.%fZ"

    def __init__(self):
        self.unused_columns: List[str] = []
        self.user_lut = self._load_mapping_metadata()

    def _load_mapping_metadata(self) -> Dict[str, str]:
        raise Exception("Must invoke a concrete class; this is the abstract one")

    def _write_user_data(self, usage_data: Dict[str, str]) -> None:
        raise Exception("Must invoke a concrete class; this is the abstract one")

    def update_user_statuses(self, user_statuses: List[Dict[str, str]]) -> None:
        usage_data: Dict[str, str] = {}
        for user in user_statuses:
            for key in list(user.keys()):
                usage_data[key] = user[key]
        self._write_user_data(usage_data)


class ServerPolicyHoldingsTracking(UsageTracking):
    """SLS implementation for policy-template holdings tracking."""

    DATA_STORAGE_PATH = "C:\\ProgramData\\National Instruments\\Shared\\"
    USER_LUT_FILENAME = "policy_user_lut.csv"
    USAGE_DATA_FILENAME = "target_permission_holdings_data.csv"

    def _update_user_mapping(self, user_mappings: List[Dict[str, str]]) -> None:
        if not user_mappings:
            return
        column_names = user_mappings[0].keys()
        with open(self.DATA_STORAGE_PATH + self.USER_LUT_FILENAME, "w", newline="") as user_lut_csv:
            writer = csv.DictWriter(user_lut_csv, fieldnames=column_names)
            writer.writeheader()
            writer.writerows(user_mappings)

    def _create_user_lut(self) -> None:
        try:
            with open(self.DATA_STORAGE_PATH + self.USER_LUT_FILENAME, "x"):
                pass
        except FileExistsError:
            print("User LUT already exists; skipping create operation.")

    def _create_data_store(self):
        self._create_user_lut()
        column_headers = ["day"]
        for i in range(self.NUM_TRACKED_USERS):
            column_headers.append(f"User_{i}")
        with open(self.DATA_STORAGE_PATH + self.USAGE_DATA_FILENAME, "x") as usage_data_csv:
            usage_data_csv.write(",".join(column_headers) + "\r\n")

    def _load_mapping_metadata(self) -> Dict[str, str]:
        lut: Dict[str, str] = {}
        id_mapping: Dict[str, str] = {}
        if not (
            os.path.exists(self.DATA_STORAGE_PATH + self.USAGE_DATA_FILENAME)
            and os.path.exists(self.DATA_STORAGE_PATH + self.USER_LUT_FILENAME)
        ):
            self._create_data_store()

        with open(self.DATA_STORAGE_PATH + self.USAGE_DATA_FILENAME, newline="") as usage_data_csv:
            reader = csv.DictReader(usage_data_csv)
            headers = reader.fieldnames or []

        with open(self.DATA_STORAGE_PATH + self.USER_LUT_FILENAME, newline="") as user_lut_csv:
            reader = csv.DictReader(user_lut_csv)
            for row in reader:
                column_name = row.get("column_name")
                user_id = row.get("id")
                if column_name and user_id:
                    lut[column_name] = user_id
                    id_mapping[user_id] = column_name

        for column in headers:
            if column != "day" and column not in lut:
                self.unused_columns.append(column)

        return id_mapping

    def _write_user_data(self, usage_data: Dict[str, str]) -> None:
        updated_user_lut: List[Dict[str, str]] = []
        column_names = ["day"]
        timestamps: List[str] = [datetime.now().strftime(self.TIME_FORMAT)]

        for user_id in usage_data.keys():
            try:
                column_name = self.user_lut[user_id]
            except KeyError:
                if not self.unused_columns:
                    raise Exception("Ran out of user tracking columns.")
                column_name = self.unused_columns.pop()
                self.user_lut[user_id] = column_name

            updated_user_lut.append({"id": user_id, "column_name": column_name})
            column_names.append(column_name)
            timestamps.append(usage_data[user_id])

        self._update_user_mapping(updated_user_lut)
        new_usage_data = pd.read_csv(self.DATA_STORAGE_PATH + self.USAGE_DATA_FILENAME, sep=",", dtype=str)
        timestamps_data = pd.DataFrame(columns=column_names, data=[timestamps], dtype=str)
        df = new_usage_data.merge(timestamps_data, how="outer")
        df.fillna("", inplace=True)
        df.to_csv(self.DATA_STORAGE_PATH + self.USAGE_DATA_FILENAME, index=False, header=True, quotechar='"')


class EnterprisePolicyHoldingsTracking(UsageTracking):
    """SLE implementation for policy-template holdings tracking."""

    DATAFRAME_NAME = "TargetPermissionHoldingsData"
    WORKSPACE_TO_USE = None

    def __init__(self) -> None:
        self.client = DataFrameClient()
        self.table_id = None
        super().__init__()

    def _create_table(self) -> None:
        columns = [Column(name="day", data_type=DataType.Timestamp, column_type=ColumnType.Index)]
        for i in range(self.NUM_TRACKED_USERS):
            columns.append(
                Column(name=f"User_{i}", data_type=DataType.Timestamp, column_type=ColumnType.Nullable)
            )
        create_table_request = CreateTableRequest(
            name=self.DATAFRAME_NAME,
            columns=columns,
            workspace=self.WORKSPACE_TO_USE,
        )
        created_table = self.client.create_table(create_table_request)
        self.table_id = created_table.id

    def _load_mapping_metadata(self) -> Dict[str, str]:
        id_mapping: Dict[str, str] = {}

        table_query = QueryTablesRequest(filter=f'name = "{self.DATAFRAME_NAME}"')
        table_details = self.client.query_tables(table_query)

        if len(table_details.tables) == 0:
            self._create_table()
            table_details = self.client.query_tables(table_query)
        elif len(table_details.tables) > 1:
            raise Exception(f"Found multiple tables with the name {self.DATAFRAME_NAME}.")

        if len(table_details.tables) == 0:
            raise Exception("Failed to create or locate target dataframe table.")

        self.table_id = table_details.tables[0].id
        for column in table_details.tables[0].columns:
            user_id = column.properties.get("id") if column.properties else None
            if user_id:
                id_mapping[user_id] = column.name
            elif column.name != "day":
                self.unused_columns.append(column.name)

        return id_mapping

    def _set_column_mapping(self, user_id: str, column_name: str) -> None:
        column_to_map = ColumnMetadataPatch(name=column_name, properties={"id": user_id})
        new_meta = ModifyTableRequest(columns=[column_to_map])
        self.client.modify_table(self.table_id, new_meta)

    def _write_user_data(self, usage_data: Dict[str, str]) -> None:
        column_names = ["day"]
        timestamps: List[str] = [datetime.now().strftime(self.TIME_FORMAT)]
        for user_id in usage_data.keys():
            try:
                column_name = self.user_lut[user_id]
            except KeyError:
                if not self.unused_columns:
                    raise Exception("Ran out of user tracking columns.")
                column_name = self.unused_columns.pop()
                self._set_column_mapping(user_id, column_name)
                self.user_lut[user_id] = column_name
            column_names.append(column_name)
            timestamps.append(usage_data[user_id])

        timestamps_data = DataFrame(columns=column_names, data=[timestamps])
        latest_data = AppendTableDataRequest(frame=timestamps_data)
        self.client.append_table_data(self.table_id, latest_data)


class ServerLatestUpdatedTracking(ServerPolicyHoldingsTracking):
    """SLS implementation for original usage tracking (latest user updated timestamps)."""

    USER_LUT_FILENAME = "user_lut.csv"
    USAGE_DATA_FILENAME = "usage_data.csv"


class EnterpriseLatestUpdatedTracking(EnterprisePolicyHoldingsTracking):
    """SLE implementation for original usage tracking (latest user updated timestamps)."""

    DATAFRAME_NAME = "UsageCalculationData"


def canonical_statement(statement: Dict[str, Any]) -> str:
    """Convert a statement dict into a stable string for exact matching."""

    def _normalize(value: Any) -> Any:
        if isinstance(value, dict):
            return {k: _normalize(value[k]) for k in sorted(value.keys())}
        if isinstance(value, list):
            normalized = [_normalize(item) for item in value]
            if all(isinstance(item, (str, int, float, bool, type(None))) for item in normalized):
                return sorted(normalized, key=lambda item: json.dumps(item, sort_keys=True))
            return normalized
        return value

    return json.dumps(_normalize(statement), sort_keys=True, separators=(",", ":"))


def find_first_list(payload: Any, candidate_keys: Set[str]) -> List[Dict[str, Any]]:
    """Find the first list of dicts in payload under commonly used keys."""
    if isinstance(payload, dict):
        for key, value in payload.items():
            if key.lower() in candidate_keys and isinstance(value, list):
                return [item for item in value if isinstance(item, dict)]
        for value in payload.values():
            result = find_first_list(value, candidate_keys)
            if result:
                return result
    elif isinstance(payload, list):
        for item in payload:
            result = find_first_list(item, candidate_keys)
            if result:
                return result
    return []


def extract_statements(payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    return find_first_list(payload, {"statements", "statement", "rules"})


def extract_string_ids(value: Any) -> Set[str]:
    """Recursively collect UUID-like strings."""
    ids: Set[str] = set()
    if isinstance(value, str):
        if re.match(r"^[0-9a-fA-F-]{32,}$", value):
            ids.add(value)
    elif isinstance(value, list):
        for item in value:
            ids.update(extract_string_ids(item))
    elif isinstance(value, dict):
        for nested in value.values():
            ids.update(extract_string_ids(nested))
    return ids


In [ ]:
# Placeholder — tracker initialization is handled in the final write cell.

In [ ]:
class HttpRouteConstants:
    """Constants for HTTP routes."""

    # Base Routes
    USER_BASE_ROUTE: str = "/niuser/v1"
    AUTH_BASE_ROUTE: str = "/niauth/v1"

    # User Routes
    QUERY_USER_DATA = "/users/query"

    # Authorization Routes
    QUERY_POLICY_TEMPLATES = [
        "/policy-templates/query",
        "/auth-templates/query",
        "/templates/query",
    ]
    GET_POLICY_TEMPLATE_BY_ID = [
        "/policy-templates/{id}",
        "/auth-templates/{id}",
        "/templates/{id}",
    ]
    QUERY_POLICIES = [
        "/policies/query",
        "/authorization-policies/query",
    ]

In [ ]:
# Import dependencies
import requests

api_key = os.getenv("SYSTEMLINK_API_KEY")
base_url = os.getenv("SYSTEMLINK_HTTP_URI").rstrip("/")

headers = {
    "accept": "application/json",
    "Content-Type": "application/json",
    "x-ni-api-key": api_key,
}


def _extract_items(payload: Any, item_keys: Set[str]) -> List[Dict[str, Any]]:
    if isinstance(payload, list):
        return [item for item in payload if isinstance(item, dict)]

    if not isinstance(payload, dict):
        return []

    for key in item_keys:
        value = payload.get(key)
        if isinstance(value, list):
            return [item for item in value if isinstance(item, dict)]

    for value in payload.values():
        if isinstance(value, list) and all(isinstance(item, dict) for item in value):
            return value

    return []


def _candidate_route_variants(route: str) -> List[str]:
    variants = [route]
    if route.endswith("/query"):
        variants.append(route[:-len("/query")])

    deduped = []
    for candidate in variants:
        if candidate not in deduped:
            deduped.append(candidate)
    return deduped


def query_paginated(route_candidates, item_keys: Set[str], take: int = 100) -> List[Dict[str, Any]]:
    """Query endpoint candidates with POST and GET fallback, including /query and non-query route variants."""
    errors: List[str] = []

    for route in route_candidates:
        for candidate in _candidate_route_variants(route):
            endpoint = f"{base_url}{HttpRouteConstants.AUTH_BASE_ROUTE}{candidate}"

            continuation_token = None
            all_items: List[Dict[str, Any]] = []
            post_supported = True

            while True:
                body = {"continuationToken": continuation_token, "take": take}
                try:
                    resp = requests.post(endpoint, headers=headers, json=body, timeout=30)
                    if resp.status_code in (404, 405):
                        post_supported = False
                        break
                    resp.raise_for_status()
                    payload = resp.json()
                except Exception as ex:
                    errors.append(f"POST {endpoint}: {ex}")
                    post_supported = False
                    break

                items = _extract_items(payload, item_keys)
                all_items.extend(items)

                continuation_token = payload.get("continuationToken") if isinstance(payload, dict) else None
                if not continuation_token:
                    break

            if post_supported and all_items:
                return all_items
            if post_supported and not all_items:
                return []

            try:
                resp = requests.get(endpoint, headers=headers, params={"take": take}, timeout=30)
                if resp.status_code in (404, 405):
                    errors.append(f"GET {endpoint}: status {resp.status_code}")
                    continue
                resp.raise_for_status()
                payload = resp.json()
                items = _extract_items(payload, item_keys)
                if items:
                    return items
                return []
            except Exception as ex:
                errors.append(f"GET {endpoint}: {ex}")

    raise Exception("Failed to query authorization endpoint candidates. " + " | ".join(errors[-8:]))


def get_template_detail(template_id: str) -> Dict[str, Any]:
    """Try template detail routes and return the first successful payload."""
    errors: List[str] = []

    for route in HttpRouteConstants.GET_POLICY_TEMPLATE_BY_ID:
        formatted = route.format(id=template_id)
        for candidate in _candidate_route_variants(formatted):
            endpoint = f"{base_url}{HttpRouteConstants.AUTH_BASE_ROUTE}{candidate}"
            try:
                resp = requests.get(endpoint, headers=headers, timeout=30)
                resp.raise_for_status()
                payload = resp.json()
                if isinstance(payload, dict):
                    return payload
            except Exception as ex:
                errors.append(f"GET {endpoint}: {ex}")

    raise Exception("Failed to retrieve policy template detail. " + " | ".join(errors[-8:]))


def get_id(record: Dict[str, Any]) -> Optional[str]:
    for key in ("id", "templateId", "policyTemplateId", "policy_id", "template_id"):
        value = record.get(key)
        if isinstance(value, str) and value:
            return value
    return None


def get_template_name(record: Dict[str, Any]) -> str:
    for key in ("name", "displayName", "templateName"):
        value = record.get(key)
        if isinstance(value, str):
            return value
    return ""


def get_template_id_from_policy(policy: Dict[str, Any]) -> Optional[str]:
    for key in ("templateId", "policyTemplateId", "template_id"):
        value = policy.get(key)
        if isinstance(value, str) and value:
            return value

    template_obj = policy.get("template")
    if isinstance(template_obj, dict):
        return get_id(template_obj)

    return None


def users_from_policy(policy: Dict[str, Any]) -> Set[str]:
    """Extract user IDs directly assigned in a policy object (best effort)."""
    matched_ids: Set[str] = set()

    def _walk(node: Any, parent_key: str = "") -> None:
        if isinstance(node, dict):
            for key, value in node.items():
                key_lower = key.lower()
                if key_lower in {"userid", "user_id", "userids", "user_ids", "users", "subjects", "principals"}:
                    extracted = extract_string_ids(value)
                    matched_ids.update(extracted)
                _walk(value, key)
        elif isinstance(node, list):
            for item in node:
                _walk(item, parent_key)

    _walk(policy)
    return matched_ids


def users_from_user_record(user: Dict[str, Any], matched_policy_ids: Set[str], matched_template_ids: Set[str]) -> bool:
    """Determine if a user record references a matched policy/template ID."""
    referenced = extract_string_ids(user)
    return bool((referenced & matched_policy_ids) or (referenced & matched_template_ids))


In [ ]:
DEFAULT_TARGET_PERMISSION_STATEMENTS: List[Dict[str, Any]] = [
    {
        "description": "",
        "resource": ["*"],
        "actions": [
            "alarm:AccessApplication",
            "alarm:Read",
            "alarm:Acknowledge",
            "asset:AccessWebUI",
            "asset:Query",
            "asset:Update",
            "dashboard:AccessApplication",
            "dashboard:ViewDashboard",
            "dataframe:AccessApplication",
            "dataframe:List",
            "dataframe:Read",
            "dataspace:AccessApplication",
            "notebookartifact:Query",
            "feed:AccessApplication",
            "feed:Read",
            "file:AccessApplication",
            "file:Download",
            "file:Query",
            "file:Update",
            "file:Upload",
            "location:Read",
            "notebookexecution:Create",
            "notebookartifact:Create",
            "notebookartifact:Update",
            "notebookexecution:Query",
            "resourcegroup:Query",
            "routine:AccessApplication",
            "routine:Query",
            "spec:Query",
            "state:AccessApplication",
            "state:Read",
            "state:Export",
            "state:ReadHistory",
            "system:AccessApplication",
            "system:ExecuteJob",
            "system:CancelJob",
            "system-job:Execute",
            "system:ReadJobs",
            "system:ReadMetadata",
            "tag:QueryTagMetadata",
            "tag:QueryTagValue",
            "taghistory:Read",
            "testmonitor:AccessApplication",
            "webapphost:AccessApplication",
            "workorder:AccessApplication",
            "workitem:AccessApplication",
            "workorder:Query",
            "workorder:Update",
            "workitem:Query",
            "workitem:Schedule",
            "workitem:Execute",
            "workitemworkflow:Query",
        ],
    },
    {
        "description": "",
        "resource": ["assets:asset"],
        "actions": [
            "comments:Create",
            "comments:Query",
            "comments:UpdateOwn",
            "comments:DeleteOwn",
            "tables:Query",
            "tables:Update",
        ],
    },
    {
        "description": "",
        "resource": ["DataSpace"],
        "actions": [
            "webapp:GetWebApp",
            "webapp:DownloadWebApp",
            "webapp:GetWebAppContent",
            "comments:Create",
            "comments:Query",
            "comments:UpdateOwn",
            "comments:DeleteOwn",
        ],
    },
    {
        "description": "",
        "resource": ["workorder:workorder", "workitem:workitem"],
        "actions": ["dynamicformfields:Query"],
    },
    {
        "description": "",
        "resource": ["Notebook"],
        "actions": [
            "webapp:GetWebApp",
            "webapp:DownloadWebApp",
            "webapp:GetWebAppContent",
        ],
    },
    {
        "description": "",
        "resource": ["*"],
        "actions": ["notification:Read", "system-package:Read", "system-unmanaged:Read"],
        "workspace": "global",
    },
    {
        "description": "",
        "resource": [
            "state.apply",
            "nisysmgmt.restart",
            "nisysmgmt.restart_if_required",
            "system.get_reboot_required_witnessed",
            "pkg.info_installed",
            "pkg.list_repos",
        ],
        "actions": ["system-job:Execute"],
    },
    {
        "description": "",
        "resource": [
            "nisysapi.get_props",
            "nisysapi.populate_props",
            "system.get_system_date",
            "system.get_system_time",
            "timezone.get_zone",
        ],
        "actions": ["system-job:Execute"],
    },
    {
        "description": "",
        "resource": ["systems:system"],
        "actions": [
            "comments:Create",
            "comments:Query",
            "comments:UpdateOwn",
            "comments:DeleteOwn",
        ],
    },
    {
        "description": "",
        "resource": ["Result", "Step"],
        "actions": ["testmonitor:Create", "testmonitor:Query", "testmonitor:Update"],
    },
    {
        "description": "",
        "resource": ["testmonitor:Result"],
        "actions": [
            "comments:Create",
            "comments:Query",
            "comments:UpdateOwn",
            "comments:DeleteOwn",
        ],
    },
    {
        "description": "",
        "resource": ["Product"],
        "actions": ["testmonitor:Query"],
    },
    {
        "description": "",
        "resource": ["testmonitor:Product"],
        "actions": [
            "comments:Create",
            "comments:Query",
            "comments:UpdateOwn",
            "comments:DeleteOwn",
            "tables:Query",
        ],
    },
    {
        "description": "",
        "resource": ["WebVI"],
        "actions": [
            "webapp:GetWebApp",
            "webapp:DownloadWebApp",
            "webapp:GetWebAppContent",
        ],
    },
    {
        "description": "",
        "resource": ["workorder:workorder"],
        "actions": [
            "dynamicformfields:Query",
            "comments:Create",
            "comments:Query",
            "comments:UpdateOwn",
            "comments:DeleteOwn",
            "tables:Query",
            "tables:Update",
        ],
    },
    {
        "description": "",
        "resource": ["workitem:workitem"],
        "actions": [
            "dynamicformfields:Query",
            "comments:Create",
            "comments:Query",
            "comments:UpdateOwn",
            "comments:DeleteOwn",
            "tables:Query",
        ],
    },
]


# The built-in SystemLink role whose permission statements we want to track.
TARGET_ROLE_NAME = "Operator"


def get_target_permission_statements(
    templates: List[Dict[str, Any]],
    role_name: str = TARGET_ROLE_NAME,
) -> List[Dict[str, Any]]:
    """Load target statements from the named SystemLink role (policy template).

    Falls back to DEFAULT_TARGET_PERMISSION_STATEMENTS if the role cannot be
    found or has no usable permission statements.
    """
    matches = [
        template
        for template in templates
        if get_template_name(template).strip().lower() == role_name.lower()
    ]

    if not matches:
        available = sorted({get_template_name(t) for t in templates if get_template_name(t)})
        print(
            f'Role "{role_name}" not found among policy templates '
            f"(available: {', '.join(available) or '(none)'}). "
            "Falling back to default target permission statements."
        )
        return DEFAULT_TARGET_PERMISSION_STATEMENTS

    if len(matches) > 1:
        print(
            f'Found multiple roles named "{role_name}"; cannot disambiguate. '
            "Falling back to default target permission statements."
        )
        return DEFAULT_TARGET_PERMISSION_STATEMENTS

    template_id = get_id(matches[0])
    statements: List[Dict[str, Any]] = []
    if template_id:
        try:
            detail = get_template_detail(template_id)
            statements = extract_statements(detail) or extract_statements(matches[0])
        except Exception as ex:
            print(f'Failed to retrieve detail for role "{role_name}": {ex}')
    else:
        statements = extract_statements(matches[0])

    if not statements:
        print(
            f'Role "{role_name}" has no usable permission statements. '
            "Falling back to default target permission statements."
        )
        return DEFAULT_TARGET_PERMISSION_STATEMENTS

    print(f'Using target permission statements from role "{role_name}".')
    return statements


all_templates = query_paginated(
    HttpRouteConstants.QUERY_POLICY_TEMPLATES,
    item_keys={"policyTemplates", "templates", "items"},
)

target_statements = get_target_permission_statements(all_templates)
target_statement_set = {canonical_statement(statement) for statement in target_statements}
if not target_statement_set:
    raise Exception("No target permission statements found.")

matching_template_ids: Set[str] = set()
for template in all_templates:
    template_id = get_id(template)
    if not template_id:
        continue

    detail = get_template_detail(template_id)
    template_statements = extract_statements(detail) or extract_statements(template)
    if not template_statements:
        continue

    template_statement_set = {canonical_statement(statement) for statement in template_statements}
    # Match criteria: any overlap between target statements and template statements.
    if target_statement_set & template_statement_set:
        matching_template_ids.add(template_id)

print(f"Matched policy template IDs ({len(matching_template_ids)}):")
for template_id in sorted(matching_template_ids):
    print(template_id)

# 3) Query policies and determine matched policy IDs
all_policies = query_paginated(
    HttpRouteConstants.QUERY_POLICIES,
    item_keys={"policies", "authorizationPolicies", "items"},
)

matched_policy_ids: Set[str] = set()
user_ids_holding_policy: Set[str] = set()

for policy in all_policies:
    policy_id = get_id(policy)
    policy_template_id = get_template_id_from_policy(policy)
    if policy_template_id in matching_template_ids:
        if policy_id:
            matched_policy_ids.add(policy_id)
        user_ids_holding_policy.update(users_from_policy(policy))

# 4) Query all users (used for both datasets)
get_users_url = f"{base_url}{HttpRouteConstants.USER_BASE_ROUTE}{HttpRouteConstants.QUERY_USER_DATA}"
continuation_token = None
all_users: List[Dict[str, Any]] = []

while True:
    body = {"continuationToken": continuation_token, "take": 100}
    query_resp = requests.post(get_users_url, headers=headers, json=body, timeout=30)
    query_resp.raise_for_status()
    payload = query_resp.json()

    users = payload.get("users", [])
    if isinstance(users, list):
        all_users.extend([user for user in users if isinstance(user, dict)])

    continuation_token = payload.get("continuationToken")
    if not continuation_token:
        break

# 5) Use user records to cross-check policy holdings
for user in all_users:
    user_id = user.get("id")
    if not isinstance(user_id, str) or not user_id:
        continue
    if users_from_user_record(user, matched_policy_ids, matching_template_ids):
        user_ids_holding_policy.add(user_id)

print(f"Users holding matched-template policies: {len(user_ids_holding_policy)}")

# 6) Build policy-holdings daily list (timestamp present => held policy today)
now_iso = datetime.now().strftime(UsageTracking.TIME_FORMAT)
user_list_policy = [{user_id: now_iso} for user_id in sorted(user_ids_holding_policy)]

# 7) Build original usage snapshot (each user's latest updated timestamp)
user_list_latest_updated = []
for user in all_users:
    user_id = user.get("id")
    updated = user.get("updated")
    if isinstance(user_id, str) and user_id and isinstance(updated, str) and updated:
        user_list_latest_updated.append({user_id: updated})

print(f"Users in latest-updated snapshot: {len(user_list_latest_updated)}")

In [ ]:
# Initialize both trackers and persist both daily snapshots independently
write_results = []

try:
    latest_updated_tracker = EnterpriseLatestUpdatedTracking()
except Exception:
    latest_updated_tracker = ServerLatestUpdatedTracking()

try:
    policy_tracker = EnterprisePolicyHoldingsTracking()
except Exception:
    policy_tracker = ServerPolicyHoldingsTracking()

latest_payload = user_list_latest_updated if "user_list_latest_updated" in globals() else []
policy_payload = user_list_policy if "user_list_policy" in globals() else []

try:
    latest_updated_tracker.update_user_statuses(latest_payload)
    write_results.append("UsageCalculationData: success")
except Exception as ex:
    write_results.append(f"UsageCalculationData: failed ({ex})")

try:
    policy_tracker.update_user_statuses(policy_payload)
    write_results.append("TargetPermissionHoldingsData: success")
except Exception as ex:
    write_results.append(f"TargetPermissionHoldingsData: failed ({ex})")

for result in write_results:
    print(result)
